# BERT MODEL

References code I follow by GreeksforGreeks 

### Step 1: Install and Import Required Libraries

In [ ]:
!pip install transformers torch

In [ ]:
from transformers import BertTokenizer, BertModel
import torch

### Step 2: Load the Pretrained BERT Model and Tokenizer

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-cased")
model = BertModel.from_pretrained("bert-base-cased")

### Step 3: Tokenize the Input Text

In [ ]:
text = "I sat by the river bank to enjoy the view."

inputs = tokenizer(
    text,
    return_tensors="pt"
)

### Step 4: Mask the Target Word

In [ ]:
masked_text = "I sat by the river [MASK] to enjoy the view."

inputs = tokenizer(
    masked_text,
    return_tensors="pt"
)

tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

### Step 5: Load BERT Masked Language Model

In [ ]:
from transformers import BertForMaskedLM

mlm_model = BertForMaskedLM.from_pretrained("bert-base-cased")
mlm_model.eval()

### Step 6: Run the Model

In [ ]:
with torch.no_grad():
    outputs = mlm_model(**inputs)

outputs.logits.shape

### Step 7: Predict the Masked Word

In [ ]:
mask_index = (inputs["input_ids"][0] == tokenizer.mask_token_id).nonzero().item()

probs = torch.softmax(outputs.logits[0, mask_index], dim=-1)
top = probs.topk(3)

for p, i in zip(top.values, top.indices):
    print(f"{tokenizer.convert_ids_to_tokens(i.item()):<8} {p:.4f}")

### Step 8: Fill the Blank

In [ ]:
predicted = tokenizer.convert_ids_to_tokens(top.indices[0].item())

print("Input :", masked_text)
print("Output:", masked_text.replace("[MASK]", predicted))